# # 020 — Pilot: đo thời gian, kiểm forward/backward 2 proposed
#
# **Mục tiêu (giai đoạn B):** trên Kaggle GPU thật, đo các số liệu để chốt `compute_budget.md`:
# 1. ResNet18 (B1): s/epoch full train, GPU memory, val accuracy sau 2 epoch (sanity).
# 2. ASF: kiểm khởi tạo ≡ ResNet18, forward/backward, 1 epoch trên subset, latency, gate hoạt động.
# 3. CMP: forward/backward, margin hoạt động đúng chiều, 1 epoch subset, prototype stats.
# 4. Latency batch-1 mọi model chính.
# 5. Xuất `pilot_report.json`.
#
# **Ràng buộc:** CHỈ dùng train/val. Không đụng test. Không smoke metrics = final metrics.
# **Nguồn:** dataset `punfake/resisc45` + split manifest từ kernel `resisc45-data-audit` (kernel_sources).

In [1]:
# ---- CONFIG ----
import os

RUN_ID = "020-pilot-v1"
MANIFEST = None  # resolve ở cell sau
DATA_ROOT = None
SUBSET_FRAC = 0.2     # phần train dùng cho run ngắn ASF/CMP
PILOT_EPOCHS_FULL = 2 # ResNet18 chạy đủ train
PILOT_EPOCHS_SUB = 1
SEED = 0

In [2]:
# ---- ENV + resolve paths + manifest ----
import glob
import json
import sys
import time

import numpy as np
import pandas as pd
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"run_id: {RUN_ID} | device: {device}")
print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| cuda:", torch.version.cuda)
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0), "| mem:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

def find_single(pattern_list: list[str], desc: str) -> str:
    for pat in pattern_list:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    raise AssertionError(f"FAIL-FAST: khong tim thay {desc}: {pattern_list}")

manifest_path = find_single(
    ["/kaggle/input/resisc45-data-audit/split_manifest.csv",
     "../../notebooks/010-data-audit/split_manifest.smoke.csv",
     "split_manifest.smoke.csv"],
    "split manifest")
DATA_ROOT = find_single(
    ["/kaggle/input/resisc45/NWPU-RESISC45",
     "../../data/raw/punfake---resisc45/NWPU-RESISC45"],
    "dataset root")
manifest = pd.read_csv(manifest_path)
manifest = manifest[manifest.split != "test"].reset_index(drop=True)  # pilot: KHÔNG test
print("manifest:", manifest_path, "| rows (train+val):", len(manifest))
print("DATA_ROOT:", DATA_ROOT)
audit_path = find_single(
    ["/kaggle/input/resisc45-data-audit/audit_report.json",
     "../../notebooks/010-data-audit/audit_report.smoke.json"],
    "audit report")
with open(audit_path) as f:
    audit = json.load(f)
print("audit run_id:", audit["run_id"], "| smoke:", audit["smoke"], "| split sha256:", audit["split_sha256"][:16])

run_id: 020-pilot-v1 | device: cpu
python: 3.12.15 | torch: 2.14.1 | cuda: None
manifest: ../../notebooks/010-data-audit/split_manifest.smoke.csv | rows (train+val): 255
DATA_ROOT: ../../data/raw/punfake---resisc45/NWPU-RESISC45
audit run_id: 010-data-audit-v1 | smoke: True | split sha256: d2cc4dd9eb852a9a


In [3]:
# ===== embedded from src/models.py (single source of logic) =====
# """Nguon logic dung chung (single source) cho moi notebook: build model, train, eval.
# Được nbmake.py embed vào notebook khi build — notebook tự chứa bản copy sinh tự động,
# KHÔNG sửa trực tiếp trong notebook; sửa file này rồi rebuild.
# """
from __future__ import annotations

import copy
import json
import math
import os
import random
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image

# ---------------------------------------------------------------- utils

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def count_params(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": int(total), "trainable": int(trainable)}


# ---------------------------------------------------------------- data


@dataclass
class TrainConfig:
    model_key: str = "resnet18"          # smallcnn | resnet18 | mobilenetv3s | efficientnetb0 | convnextt | asf | cmp
    num_classes: int = 45
    seed: int = 0
    epochs: int = 15
    batch_size: int = 128
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 1e-4
    warmup_frac: float = 0.05
    early_stop_patience: int = 5
    amp: bool = True
    num_workers: int = 4
    input_size: int = 256
    # ASF
    asf_fixed_fusion: bool = False        # A1 (True) vs A2 (False)
    asf_feature_dim: int = 128
    # CMP
    cmp_n_proto: int = 1                  # C1/C2: 1 hoặc 2, 4
    cmp_confusion: bool = False           # C3
    cmp_conf_margin: float = 0.3
    cmp_conf_weight: float = 0.5
    cmp_diversity: bool = False           # C4
    cmp_div_weight: float = 0.01
    cmp_embed_dim: int = 512
    # chung
    label_smoothing: float = 0.0
    extra: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)


class ResiscDataset(Dataset):
    """Đọc ảnh theo manifest (image_id relpath từ root NWPU-RESISC45)."""

    def __init__(self, manifest: pd.DataFrame, root: str, train: bool, input_size: int = 256):
        self.items = manifest.reset_index(drop=True)
        self.root = root
        self.train = train
        self.input_size = input_size

    def __len__(self) -> int:
        return len(self.items)

    def _path(self, i: int) -> str:
        r = self.items.iloc[i]
        return os.path.join(self.root, r.image_id)

    def __getitem__(self, i: int):
        img = Image.open(self._path(i)).convert("RGB")
        img = train_augment(img, self.input_size) if self.train else eval_augment(img, self.input_size)
        return img, int(self.items.iloc[i].class_index)


def train_augment(img: Image.Image, size: int) -> Image.Image:
    """Augmentation chung (thuần PIL, tránh phụ thuộc torchvision.transforms)."""
    import torchvision.transforms.functional as TF

    # RandomResizedCrop(scale 0.7–1.0) + hflip + vflip
    w, h = img.size
    scale = random.uniform(0.7, 1.0)
    ratio = random.uniform(3 / 4, 4 / 3)
    area = w * h * scale
    cw = min(w, int(round(math.sqrt(area * ratio))))
    ch = min(h, int(round(math.sqrt(area / ratio))))
    x = random.randint(0, w - cw)
    y = random.randint(0, h - ch)
    img = img.crop((x, y, x + cw, y + ch)).resize((size, size), Image.BILINEAR)
    if random.random() < 0.5:
        img = TF.hflip(img)
    if random.random() < 0.5:
        img = TF.vflip(img)
    return TF.normalize(TF.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)


def eval_augment(img: Image.Image, size: int) -> "torch.Tensor":
    import torchvision.transforms.functional as TF

    img = img.resize((size, size), Image.BILINEAR)
    return TF.normalize(TF.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)


def make_loaders(manifest: pd.DataFrame, root: str, cfg: TrainConfig):
    tr = manifest[manifest.split == "train"]
    va = manifest[manifest.split == "val"]
    dl_train = DataLoader(ResiscDataset(tr, root, True, cfg.input_size), batch_size=cfg.batch_size,
                          shuffle=True, num_workers=cfg.num_workers, pin_memory=True, drop_last=False)
    dl_val = DataLoader(ResiscDataset(va, root, False, cfg.input_size), batch_size=cfg.batch_size,
                        shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return dl_train, dl_val


# ---------------------------------------------------------------- baselines


def build_model(cfg: TrainConfig, pretrained: bool = True) -> nn.Module:
    """B0–B4 và 2 proposed. pretrained=False chỉ dùng cho smoke shape-check."""
    import torchvision

    if cfg.model_key == "smallcnn":
        return SmallCNN(cfg.num_classes)
    if cfg.model_key == "resnet18":
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, cfg.num_classes)
        return m
    if cfg.model_key == "mobilenetv3s":
        m = torchvision.models.mobilenet_v3_small(weights=torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, cfg.num_classes)
        return m
    if cfg.model_key == "efficientnetb0":
        m = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, cfg.num_classes)
        return m
    if cfg.model_key == "convnextt":
        m = torchvision.models.convnext_tiny(weights=torchvision.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, cfg.num_classes)
        return m
    if cfg.model_key == "asf":
        return ASFResNet18(cfg.num_classes, pretrained=pretrained, fixed_fusion=cfg.asf_fixed_fusion,
                           feature_dim=cfg.asf_feature_dim)
    if cfg.model_key == "cmp":
        return CMPResNet18(cfg.num_classes, pretrained=pretrained, n_proto=cfg.cmp_n_proto,
                           embed_dim=cfg.cmp_embed_dim, confusion=cfg.cmp_confusion,
                           conf_margin=cfg.cmp_conf_margin, diversity=cfg.cmp_diversity)
    raise ValueError(f"unknown model_key: {cfg.model_key}")


class SmallCNN(nn.Module):
    """B0 — CNN nhỏ train từ đầu, cấu trúc hợp lý (3 block, BN, GAP)."""

    def __init__(self, num_classes: int = 45):
        super().__init__()
        ch = [3, 32, 64, 128]
        layers = []
        for a, b in zip(ch[:-1], ch[1:]):
            layers += [nn.Conv2d(a, b, 3, padding=1, bias=False), nn.BatchNorm2d(b), nn.ReLU(inplace=True),
                       nn.Conv2d(b, b, 3, padding=1, bias=False), nn.BatchNorm2d(b), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1))


# ---------------------------------------------------------------- ASF-ResNet18


def haar_dwt2(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Haar DWT 2D level-1 trên tensor (B,C,H,W), H/W chẵn (crop trước nếu lẻ).
    Trả về (LL, LH, HL, HH) — mỗi cái (B,C,H/2,W/2)."""
    if x.shape[-2] % 2 or x.shape[-1] % 2:
        x = x[..., : x.shape[-2] // 2 * 2, : x.shape[-1] // 2 * 2]
    a = x[..., 0::2, 0::2]
    b = x[..., 0::2, 1::2]
    c = x[..., 1::2, 0::2]
    d = x[..., 1::2, 1::2]
    ll = (a + b + c + d) / 4.0
    lh = (a + b - c - d) / 4.0
    hl = (a - b + c - d) / 4.0
    hh = (a - b - c + d) / 4.0
    return ll, lh, hl, hh


class ASFResNet18(nn.Module):
    """Adaptive Scale–Frequency ResNet18.

    - Giữ backbone ResNet18 pretrained NGUYÊN VẸN (không thay conv nào).
    - Lấy feature map layer2 (128ch, 32×32) và layer4 (512ch, 16×16) → Haar DWT → 8 thành phần.
    - Mỗi thành phần: 1×1 conv → F dims → BN+ReLU → GAP → vector f_i.
    - Gate (theo ảnh): MLP(8F→8) softmax; hoặc fixed_fusion=True: softmax over learnable logits (A1).
    - Fused = Σ w_i f_i → Linear zero-init (F→512) → residual cộng vào GAP(layer4) → head.
    - Zero-init residual ⇒ lúc đầu model ≡ ResNet18 (bảo toàn biểu diễn pretrained).
    """

    def __init__(self, num_classes: int = 45, pretrained: bool = True, fixed_fusion: bool = False,
                 feature_dim: int = 128):
        super().__init__()
        import torchvision

        bb = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.fixed_fusion = fixed_fusion
        self.feature_dim = feature_dim
        self.n_components = 8  # 2 stage × 4 subband
        F_ = feature_dim
        self.proj = nn.ModuleList([nn.Sequential(nn.Conv2d(c, F_, 1, bias=False), nn.BatchNorm2d(F_),
                                                 nn.ReLU(inplace=True)) for c in (128, 128, 128, 128, 512, 512, 512, 512)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        if fixed_fusion:
            self.fixed_logits = nn.Parameter(torch.zeros(self.n_components))
        else:
            self.gate = nn.Linear(F_ * self.n_components, self.n_components)
        self.out_proj = nn.Linear(F_, 512)
        nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)
        self.head = nn.Linear(512, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        h1 = self.layer1(h)
        f2 = self.layer2(h1)     # 128ch
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)     # 512ch
        feats = []
        for i, fm in enumerate((f2, f2, f2, f2, f4, f4, f4, f4)):
            # thành phần i%4==0: LL ≡ full feature map trước khi tách (thông tin low-freq tổng thể);
            # i%4∈{1,2,3}: LH/HL/HH từ Haar DWT level-1
            sub = fm if i % 4 == 0 else haar_dwt2(fm)[i % 4]
            feats.append(self.pool(self.proj[i](sub)).flatten(1))  # (B, F)
        stack = torch.stack(feats, dim=1)                          # (B, 8, F)
        if self.fixed_fusion:
            w = torch.softmax(self.fixed_logits, dim=0)
            w = w.view(1, -1, 1).expand_as(stack)
        else:
            w = torch.softmax(self.gate(stack.flatten(1)), dim=-1).unsqueeze(-1)  # (B, 8, 1)
        fused = (stack * w).sum(dim=1)                             # (B, F)
        backbone_vec = self.pool(f4).flatten(1)                    # (B, 512)
        return self.head(backbone_vec + self.out_proj(fused))

    def gate_weights(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về (B, 8) để phân tích phân bố gate (chỉ dùng khi fixed_fusion=False)."""
        assert not self.fixed_fusion
        was_training = self.training
        self.eval()
        with torch.no_grad():
            h = self.stem(x)
            h1 = self.layer1(h)
            f2 = self.layer2(h1)
            f4 = self.layer4(self.layer3(f2))
            feats = []
            for i, fm in enumerate((f2, f2, f2, f2, f4, f4, f4, f4)):
                sub = fm if i % 4 == 0 else haar_dwt2(fm)[i % 4]
                feats.append(self.pool(self.proj[i](sub)).flatten(1))
            w = torch.softmax(self.gate(torch.stack(feats, 1).flatten(1)), -1)
        if was_training:
            self.train()
        return w


# ---------------------------------------------------------------- CMP-ResNet18


class CMPResNet18(nn.Module):
    """Confusion-guided Multi-Prototype ResNet18.

    - Backbone ResNet18 pretrained → GAP → embed Linear → L2 normalize.
    - K prototype/lớp (L2 normalize) → sim (B, C, K) → logits = logsumexp(κ·sim, K) − log K
      (chuẩn hóa theo K, κ = logit_scale học được, init ln(10)).
    - Confusion margin (có hướng, chỉ trong loss): target=a → logit_b −= m·w[a,b] với (a,b)
      lấy từ confusion graph train-only; w ∈ [0,1].
    - Diversity reg (tùy chọn): mean max-cos giữa prototype cùng lớp.
    """

    def __init__(self, num_classes: int = 45, pretrained: bool = True, n_proto: int = 1,
                 embed_dim: int = 512, confusion: bool = False, conf_margin: float = 0.3,
                 diversity: bool = False):
        super().__init__()
        import torchvision

        bb = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.backbone = nn.Sequential(*list(bb.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.embed = nn.Linear(512, embed_dim)
        self.C, self.K = num_classes, n_proto
        self.prototypes = nn.Parameter(torch.randn(num_classes * n_proto, embed_dim) * 0.02)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
        self.confusion = confusion
        self.conf_margin = conf_margin
        self.diversity = diversity
        self.register_buffer("conf_w", torch.zeros(num_classes, num_classes))  # set_confusion() ghi vào đây

    def set_confusion(self, w: np.ndarray) -> None:
        assert w.shape == (self.C, self.C) and float(w.min()) >= 0.0 and float(w.max()) <= 1.0
        self.conf_w.copy_(torch.from_numpy(w.astype(np.float32)))

    def embeddings(self, x: torch.Tensor) -> torch.Tensor:
        z = self.pool(self.backbone(x)).flatten(1)
        return F.normalize(self.embed(z), dim=-1)

    def logits_from_emb(self, e: torch.Tensor, apply_margin: bool = False, labels: torch.Tensor | None = None) -> torch.Tensor:
        p = F.normalize(self.prototypes, dim=-1)
        sim = (e @ p.T).view(-1, self.C, self.K)
        kappa = self.logit_scale.clamp(max=math.log(100.0))
        logits = torch.logsumexp(sim * kappa, dim=2) - math.log(self.K)
        if apply_margin and self.confusion and labels is not None:
            w = self.conf_w[labels]                     # (B, C): w[a, b] trên hàng a
            w = w * (1.0 - F.one_hot(labels, self.C).float())
            logits = logits - self.conf_margin * w
        return logits

    def forward(self, x: torch.Tensor, labels: torch.Tensor | None = None) -> torch.Tensor:
        return self.logits_from_emb(self.embeddings(x), apply_margin=True, labels=labels)

    def diversity_loss(self) -> torch.Tensor:
        p = F.normalize(self.prototypes, dim=-1).view(self.C, self.K, -1)
        if self.K < 2:
            return torch.zeros((), device=p.device)
        cos = torch.bmm(p, p.transpose(1, 2))       # (C, K, K)
        eye = torch.eye(self.K, device=p.device).unsqueeze(0)
        return ((cos - eye).abs().sum(dim=(1, 2)) / (self.K * (self.K - 1))).mean()

    def prototype_usage(self, loader: DataLoader, device: str, max_batches: int | None = None) -> dict:
        """Thống kê usage/khoảng cách prototype (phát hiện collapse)."""
        self.eval()
        assign = torch.zeros(self.C, self.K, dtype=torch.long)
        n_seen = 0
        with torch.no_grad():
            for bi, (x, y) in enumerate(loader):
                if max_batches and bi >= max_batches:
                    break
                e = self.embeddings(x.to(device))
                p = F.normalize(self.prototypes, dim=-1).view(self.C, self.K, -1)
                pc = p[y]                            # (B, K, D) prototype của lớp đúng
                sim = torch.einsum("bd,bkd->bk", e, pc)
                for cls, proto in zip(y, sim.argmax(1)):
                    assign[int(cls), int(proto)] += 1
                n_seen += len(y)
        inter = []
        with torch.no_grad():
            p = F.normalize(self.prototypes, dim=-1).view(self.C, self.K, -1)
            if self.K > 1:
                pc = F.normalize(p, dim=-1)
                cos = torch.bmm(pc, pc.transpose(1, 2))
                eye = torch.eye(self.K, device=cos.device).bool().unsqueeze(0)
                inter = cos.masked_fill(eye, 0).amax(dim=(1, 2)).tolist()
        return {"n_seen": n_seen, "assign_per_class_proto": assign.tolist(),
                "unused_proto_classes": int((assign.sum(1) == 0).sum()),
                "max_inter_proto_cos_mean": float(np.mean(inter)) if inter else None}


# ---------------------------------------------------------------- confusion graph (train-only)


def build_confusion_graph(train_manifest: pd.DataFrame, root: str, graph_frac: float = 0.10,
                          seed: int = 0, probe_epochs: int = 6, device: str = "cuda",
                          min_count: int = 3, num_classes: int = 45,
                          num_workers: int = 4) -> tuple[np.ndarray, dict]:
    """Probe model train trên (1−graph_frac) của TRAIN, dự đoán trên phần graph còn lại (cùng train).
    KHÔNG dùng val/test. Trả về w (C,C) có hướng: w[a,b] = n_ab / max_row, cặp n_ab < min_count → 0.
    """
    rng = np.random.default_rng(seed)
    fit_df, graph_df = [], []
    for cls, gdf in train_manifest.groupby("class_name"):
        idx = rng.permutation(len(gdf))
        n_graph = max(1, int(round(len(gdf) * graph_frac)))
        take = gdf.iloc[idx[:n_graph]]
        graph_df.append(take)
        fit_df.append(gdf.iloc[idx[n_graph:]])
    fit_df, graph_df = pd.concat(fit_df), pd.concat(graph_df)

    pc = TrainConfig(model_key="resnet18", num_classes=num_classes, epochs=probe_epochs, seed=seed,
                     num_workers=num_workers)
    probe = build_model(pc)
    dl_fit = DataLoader(ResiscDataset(fit_df, root, True, pc.input_size), batch_size=pc.batch_size, shuffle=True,
                        num_workers=num_workers)
    dl_graph = DataLoader(ResiscDataset(graph_df, root, False, pc.input_size), batch_size=pc.batch_size, shuffle=False,
                          num_workers=num_workers)
    probe = train_model(probe, pc, dl_fit, None, device, verbose=False)[0]  # không early-stop (val None)

    n = np.zeros((num_classes, num_classes), dtype=np.float64)
    probe.eval()
    with torch.no_grad():
        for x, y in dl_graph:
            pred = probe(x.to(device)).argmax(1).cpu().numpy()
            for a, b in zip(y.numpy(), pred):
                n[a, b] += 1
    off = n.copy()
    np.fill_diagonal(off, 0.0)
    row_max = off.max(axis=1, keepdims=True)
    w = np.where((off >= min_count) & (row_max > 0), off / np.maximum(row_max, 1e-9), 0.0)
    info = {"probe_epochs": probe_epochs, "n_fit": int(len(fit_df)), "n_graph": int(len(graph_df)),
            "n_pairs_active": int((w > 0).sum()), "max_n": int(off.max()),
            "confusion_rate_graph": float(off.sum() / max(len(graph_df), 1))}
    return w.astype(np.float32), info


# ---------------------------------------------------------------- train / eval


def _param_groups(model: nn.Module, cfg: TrainConfig) -> list[dict]:
    """Backbone lr thấp hơn head lr — pretrained conv là backbone (mọi model đều có .backbone/.stem...)."""
    head_ids = set()
    for name, _p in model.named_parameters():
        if any(k in name for k in ("head", "fc", "classifier", "embed", "prototypes", "logit_scale",
                                   "proj", "gate", "out_proj", "fixed_logits")):
            head_ids.add(name)
    head, backbone = [], []
    for name, p in model.named_parameters():
        (head if name in head_ids else backbone).append(p)
    return [{"params": backbone, "lr": cfg.lr_backbone}, {"params": head, "lr": cfg.lr_head}]


def train_model(model: nn.Module, cfg: TrainConfig, dl_train: DataLoader, dl_val: DataLoader | None,
                device: str, verbose: bool = True, state_out: str | None = None) -> tuple[nn.Module, dict]:
    """Train với AMP + cosine + warmup + early stop (val macro-F1). Trả về (best_model, history)."""
    model.to(device)
    opt = torch.optim.AdamW(_param_groups(model, cfg), weight_decay=cfg.weight_decay)
    steps_per_epoch = max(len(dl_train), 1)
    total_steps = cfg.epochs * steps_per_epoch
    warmup = max(1, int(cfg.warmup_frac * total_steps))

    def lr_lambda(step: int) -> float:
        if step < warmup:
            return step / warmup
        p = (step - warmup) / max(total_steps - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and device.startswith("cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    history, best = [], {"f1": -1.0, "state": None, "epoch": -1}
    t_train0 = time.time()
    bad_epochs = 0
    for ep in range(cfg.epochs):
        model.train()
        ep_loss, ep_n, t0 = 0.0, 0, time.time()
        for x, y in dl_train:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.amp and device.startswith("cuda")):
                logits = model(x, y) if isinstance(model, CMPResNet18) else model(x)
                loss = crit(logits, y)
                if isinstance(model, CMPResNet18) and model.diversity:
                    loss = loss + cfg.cmp_div_weight * model.diversity_loss()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            ep_loss += loss.item() * len(y)
            ep_n += len(y)
        entry = {"epoch": ep, "train_loss": ep_loss / max(ep_n, 1), "epoch_sec": round(time.time() - t0, 2),
                 "lr": sched.get_last_lr()[0]}
        if dl_val is not None:
            m = evaluate(model, dl_val, device)
            entry.update({"val_acc": m["acc"], "val_macro_f1": m["macro_f1"]})
            if m["macro_f1"] > best["f1"]:
                best.update({"f1": m["macro_f1"], "epoch": ep,
                             "state": copy.deepcopy(model.state_dict())})
                bad_epochs = 0
                if state_out:
                    torch.save({"state_dict": best["state"], "config": cfg.to_dict(), "epoch": ep}, state_out)
            else:
                bad_epochs += 1
        history.append(entry)
        if verbose:
            print(json.dumps(entry))
        if dl_val is not None and bad_epochs >= cfg.early_stop_patience:
            if verbose:
                print(f"early stop @ epoch {ep} (best {best['epoch']})")
            break
    train_time = time.time() - t_train0
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    hist_summary = {"train_time_sec": round(train_time, 1), "executed_epochs": len(history),
                    "best_epoch": best["epoch"], "best_val_macro_f1": None if best["f1"] < 0 else best["f1"],
                    "history": history}
    return model, hist_summary


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> dict:
    """Full metrics: acc + macro P/R/F1 + per-class + confusion matrix."""
    from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

    model.to(device).eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.amp.autocast("cuda", enabled=device.startswith("cuda")):
            logits = model(x.to(device))
        ps.append(logits.argmax(1).cpu())
        ys.append(y)
    y_true, y_pred = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    pp, rr, ff, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    return {"acc": float((y_true == y_pred).mean()), "macro_precision": float(p), "macro_recall": float(r),
            "macro_f1": float(f1),
            "per_class": {"precision": pp.tolist(), "recall": rr.tolist(), "f1": ff.tolist(), "support": sup.tolist()},
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(), "n": int(len(y_true))}


@torch.inference_mode()
def measure_latency(model: nn.Module, device: str, input_size: int = 256, n_warmup: int = 10,
                    n_iter: int = 50) -> dict:
    """Latency model-only, batch 1, median/p95 (ms). model.eval() + cuda sync + warmup."""
    model.to(device).eval()
    x = torch.randn(1, 3, input_size, input_size, device=device)
    for _ in range(n_warmup):
        model(x)
    if device.startswith("cuda"):
        torch.cuda.synchronize()
    times = []
    for _ in range(n_iter):
        if device.startswith("cuda"):
            t0 = torch.cuda.Event(enable_timing=True)
            t1 = torch.cuda.Event(enable_timing=True)
            t0.record()
            model(x)
            t1.record()
            torch.cuda.synchronize()
            times.append(t0.elapsed_time(t1))
        else:
            t0 = time.perf_counter()
            model(x)
            times.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(times)), "p95_ms": float(np.percentile(times, 95)),
            "n_iter": n_iter, "batch": 1}


In [4]:
# ---- Dataset/loaders kiểm tra tốc độ data loader ----
set_seed(SEED)
full_train = manifest[manifest.split == "train"].reset_index(drop=True)
NUM_WORKERS = 0 if sys.platform == "darwin" else 4  # macOS smoke local: 0 worker (spawn hay crash); Kaggle Linux: 4
cfg_probe = TrainConfig(num_classes=45, seed=SEED, num_workers=NUM_WORKERS, batch_size=128)
dl_tr, dl_va = make_loaders(manifest, DATA_ROOT, cfg_probe)  # truyền train+val; make_loaders tự lọc split
t0 = time.time()
nb = 0
for x, y in dl_tr:
    nb += 1
    if nb == 8:
        break
t_loader = (time.time() - t0) / 8
print(f"data loader: {t_loader:.3f}s/batch (128 ảnh) | train imgs: {len(dl_tr.dataset)} | val: {len(dl_va.dataset)}")

/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


data loader: 0.092s/batch (128 ảnh) | train imgs: 225 | val: 30


In [5]:
# ---- B1 ResNet18: 2 epoch FULL train/val — đo s/epoch + memory ----
set_seed(SEED)
cfg_b1 = TrainConfig(model_key="resnet18", num_classes=45, seed=SEED, epochs=PILOT_EPOCHS_FULL)
b1 = build_model(cfg_b1)
params_b1 = count_params(b1)
torch.cuda.reset_peak_memory_stats() if device == "cuda" else None
b1, hist_b1 = train_model(b1, cfg_b1, dl_tr, dl_va, device)
peak_mem = torch.cuda.max_memory_allocated() / 2**30 if device == "cuda" else None
m_b1 = evaluate(b1, dl_va, device)
lat_b1 = measure_latency(b1, device, n_iter=10 if device == "cpu" else 50)
print("B1 params:", params_b1, "| val:", {k: m_b1[k] for k in ("acc", "macro_f1")},
      "| sec/epoch:", [h["epoch_sec"] for h in hist_b1["history"]], "| peak mem GiB:", peak_mem,
      "| latency:", lat_b1)

/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{"epoch": 0, "train_loss": 3.8286044385698106, "epoch_sec": 10.06, "lr": 7.500000000000001e-05, "val_acc": 0.26666666666666666, "val_macro_f1": 0.15869844441273015}


/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{"epoch": 1, "train_loss": 2.449503560596042, "epoch_sec": 9.71, "lr": 0.0, "val_acc": 0.36666666666666664, "val_macro_f1": 0.31226551226551225}


/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


B1 params: {'total': 11199597, 'trainable': 11199597} | val: {'acc': 0.36666666666666664, 'macro_f1': 0.31226551226551225} | sec/epoch: [10.06, 9.71] | peak mem GiB: None | latency: {'median_ms': 8.865958500337001, 'p95_ms': 9.247350300574908, 'n_iter': 10, 'batch': 1}


In [6]:
# ---- ASF: kiểm khởi tạo ≡ ResNet18 + forward/backward + subset epoch ----
set_seed(SEED)
cfg_asf = TrainConfig(model_key="asf", num_classes=45, seed=SEED, epochs=PILOT_EPOCHS_SUB)
asf = build_model(cfg_asf)
params_asf = count_params(asf)
# (a) zero-init ⇒ đầu ra ≡ ResNet18 fresh: so với ResNet18 CHƯA train (b1 đã train,
#     BN running stats đổi → không dùng), copy head để khác biệt chỉ còn residual branch
xb, yb = next(iter(dl_tr))
ref_fresh = build_model(TrainConfig(model_key="resnet18", num_classes=45, seed=SEED))
asf.eval(); ref_fresh.eval()
asf.head.load_state_dict(ref_fresh.fc.state_dict())
with torch.no_grad():
    l_asf, l_ref = asf(xb[:16].to(device)), ref_fresh(xb[:16].to(device))
init_diff = float((l_asf - l_ref).abs().max())
print(f"ASF init vs ResNet18-fresh logits max|Δ| = {init_diff:.4e} (mong đợi ≈ 0 do zero-init residual)")
# (b) gate weights theo ảnh có khác nhau giữa 2 ảnh không (hoạt động)
asf.train()
gw = asf.gate_weights(xb[:4].to(device))
gate_spread = float(gw.var(dim=0).mean())
print("gate shape:", tuple(gw.shape), "| gate var trung bình giữa ảnh:", f"{gate_spread:.2e}")
# (c) 1 epoch subset
sub_train = full_train.groupby("class_name").sample(frac=SUBSET_FRAC, random_state=SEED).reset_index(drop=True)
dl_sub = torch.utils.data.DataLoader(ResiscDataset(sub_train, DATA_ROOT, True, cfg_asf.input_size),
                                     batch_size=128, shuffle=True, num_workers=NUM_WORKERS)
set_seed(SEED)
asf = build_model(cfg_asf)
asf, hist_asf = train_model(asf, cfg_asf, dl_sub, dl_va, device)
lat_asf = measure_latency(asf, device, n_iter=10 if device == "cpu" else 50)
print("ASF params:", params_asf, "| subset epoch sec:", [h["epoch_sec"] for h in hist_asf["history"]],
      "| val:", hist_asf.get("best_val_macro_f1"), "| latency:", lat_asf)

/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


ASF init vs ResNet18-fresh logits max|Δ| = 0.0000e+00 (mong đợi ≈ 0 do zero-init residual)
gate shape: (4, 8) | gate var trung bình giữa ảnh: 2.39e-05


{"epoch": 0, "train_loss": 4.107616901397705, "epoch_sec": 1.99, "lr": 0.0001, "val_acc": 0.0, "val_macro_f1": 0.0}


ASF params: {'total': 11603573, 'trainable': 11603573} | subset epoch sec: [1.99] | val: 0.0 | latency: {'median_ms': 9.944041999915498, 'p95_ms': 10.267868599657959, 'n_iter': 10, 'batch': 1}


In [7]:
# ---- CMP: forward/backward + margin chiều đúng + subset epoch ----
set_seed(SEED)
w_fake = np.zeros((45, 45), dtype=np.float32)
w_fake[0, 40] = 0.8  # giả lập cặp dễ nhầm (chỉ để test cơ chế margin)
cfg_cmp = TrainConfig(model_key="cmp", num_classes=45, seed=SEED, epochs=PILOT_EPOCHS_SUB, cmp_n_proto=2, cmp_confusion=True)
cmp_model = build_model(cfg_cmp)
cmp_model.set_confusion(w_fake)
params_cmp = count_params(cmp_model)
# margin chiều đúng: ảnh label=0 → logit class 40 phải giảm khi apply margin
cmp_model.eval()
with torch.no_grad():
    e = cmp_model.embeddings(xb[:8].to(device))
    y_test = torch.zeros(8, dtype=torch.long)
    l_margin = cmp_model.logits_from_emb(e, apply_margin=True, labels=y_test)
    l_plain = cmp_model.logits_from_emb(e, apply_margin=False, labels=y_test)
delta_40 = float((l_margin[:, 40] - l_plain[:, 40]).mean())
print(f"CMP margin Δlogit[class40, label0] = {delta_40:.4f} (mong đợi < 0 = −0.3·0.8·?)")
print("CMP params:", params_cmp)
# subset epoch (không confusion — graph chỉ đo mini ở cell sau)
cfg_cmp_train = TrainConfig(model_key="cmp", num_classes=45, seed=SEED, epochs=PILOT_EPOCHS_SUB, cmp_n_proto=2)
set_seed(SEED)
cmp_model = build_model(cfg_cmp_train)
cmp_model, hist_cmp = train_model(cmp_model, cfg_cmp_train, dl_sub, dl_va, device)
usage = cmp_model.prototype_usage(dl_sub, device, max_batches=3)
lat_cmp = measure_latency(cmp_model, device, n_iter=10 if device == "cpu" else 50)
print("CMP subset epoch sec:", [h["epoch_sec"] for h in hist_cmp["history"]], "| val F1:", hist_cmp.get("best_val_macro_f1"))
print("prototype usage:", usage, "| latency:", lat_cmp)

CMP margin Δlogit[class40, label0] = -0.2400 (mong đợi < 0 = −0.3·0.8·?)
CMP params: {'total': 11485249, 'trainable': 11485249}


/Users/bangdang/Project/kaggle-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{"epoch": 0, "train_loss": 3.8340563774108887, "epoch_sec": 1.83, "lr": 0.0001, "val_acc": 0.0, "val_macro_f1": 0.0}


CMP subset epoch sec: [1.83] | val F1: 0.0
prototype usage: {'n_seen': 45, 'assign_per_class_proto': [[2, 7], [7, 2], [6, 3], [0, 9], [4, 5], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0]], 'unused_proto_classes': 40, 'max_inter_proto_cos_mean': 0.020970754904879464} | latency: {'median_ms': 8.782479500041518, 'p95_ms': 9.069715100031317, 'n_iter': 10, 'batch': 1}


In [8]:
# ---- Đo mini confusion graph (chỉ đo THỜI GIAN, không dùng graph này cho final) ----
t0 = time.time()
probe_train = full_train.groupby("class_name").head(40).reset_index(drop=True)  # mini: 40 ảnh/lớp chỉ để đo thời gian
w_probe, graph_info = build_confusion_graph(probe_train, DATA_ROOT, graph_frac=0.25, seed=SEED,
                                            probe_epochs=1, device=device, num_classes=45, num_workers=NUM_WORKERS)
graph_time = time.time() - t0
print(f"mini graph (45×40 ảnh, 1 epoch): {graph_time:.1f}s | active pairs: {graph_info['n_pairs_active']}")
print("→ ngoại suy chi phí graph thực (full train, 6 epoch probe) ở report")

mini graph (45×40 ảnh, 1 epoch): 8.6s | active pairs: 3
→ ngoại suy chi phí graph thực (full train, 6 epoch probe) ở report


In [9]:
# ---- Latency các baseline còn lại (không train) ----
lats = {"resnet18": lat_b1, "asf": lat_asf, "cmp": lat_cmp}
for key in ("smallcnn", "mobilenetv3s", "efficientnetb0", "convnextt"):
    set_seed(SEED)
    m = build_model(TrainConfig(model_key=key, num_classes=45, seed=SEED))
    lats[key] = measure_latency(m, device, n_iter=10 if device == "cpu" else 50)
    print(key, "latency:", lats[key], "params:", count_params(m))

smallcnn latency: {'median_ms': 13.684562499747699, 'p95_ms': 13.991008150151174, 'n_iter': 10, 'batch': 1} params: {'total': 293261, 'trainable': 293261}


mobilenetv3s latency: {'median_ms': 29.537437499584485, 'p95_ms': 30.138312649887666, 'n_iter': 10, 'batch': 1} params: {'total': 1563981, 'trainable': 1563981}


efficientnetb0 latency: {'median_ms': 86.78189600050246, 'p95_ms': 88.00559404990054, 'n_iter': 10, 'batch': 1} params: {'total': 4065193, 'trainable': 4065193}


convnextt latency: {'median_ms': 234.79918700058988, 'p95_ms': 260.6945828501466, 'n_iter': 10, 'batch': 1} params: {'total': 27854733, 'trainable': 27854733}


In [10]:
# ---- Xuất pilot_report.json ----
report = {
    "run_id": RUN_ID, "device": device, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "manifest": os.path.basename(manifest_path), "split_sha256": audit["split_sha256"],
    "audit_smoke": audit["smoke"],
    "resnet18_full": {"params": params_b1, "sec_per_epoch": [h["epoch_sec"] for h in hist_b1["history"]],
                      "train_time_sec": hist_b1["train_time_sec"], "val": {k: m_b1[k] for k in ("acc", "macro_f1")},
                      "peak_mem_gib": peak_mem},
    "asf": {"params": params_asf, "init_max_logit_diff": init_diff, "gate_var": gate_spread,
            "subset_epoch_sec": [h["epoch_sec"] for h in hist_asf["history"]],
            "best_val_macro_f1_subset": hist_asf.get("best_val_macro_f1")},
    "cmp": {"params": params_cmp, "margin_delta_logit": delta_40,
            "subset_epoch_sec": [h["epoch_sec"] for h in hist_cmp["history"]],
            "best_val_macro_f1_subset": hist_cmp.get("best_val_macro_f1"), "prototype_usage": usage},
    "graph": {"mini_time_sec": round(graph_time, 1), "info": graph_info},
    "latency_batch1": lats,
    "loader_sec_per_batch128": round(t_loader, 4),
}
with open("pilot_report.json", "w") as f:
    json.dump(report, f, indent=1)
print(json.dumps(report, indent=1, ensure_ascii=False))

{
 "run_id": "020-pilot-v1",
 "device": "cpu",
 "gpu": "cpu",
 "torch": "2.14.1",
 "cuda": null,
 "manifest": "split_manifest.smoke.csv",
 "split_sha256": "d2cc4dd9eb852a9aa788edab3ff9293052e9a4062eba427bc06ee4105a8b5534",
 "audit_smoke": true,
 "resnet18_full": {
  "params": {
   "total": 11199597,
   "trainable": 11199597
  },
  "sec_per_epoch": [
   10.06,
   9.71
  ],
  "train_time_sec": 21.5,
  "val": {
   "acc": 0.36666666666666664,
   "macro_f1": 0.31226551226551225
  },
  "peak_mem_gib": null
 },
 "asf": {
  "params": {
   "total": 11603573,
   "trainable": 11603573
  },
  "init_max_logit_diff": 0.0,
  "gate_var": 2.391287671343889e-05,
  "subset_epoch_sec": [
   1.99
  ],
  "best_val_macro_f1_subset": 0.0
 },
 "cmp": {
  "params": {
   "total": 11485249,
   "trainable": 11485249
  },
  "margin_delta_logit": -0.24000000953674316,
  "subset_epoch_sec": [
   1.83
  ],
  "best_val_macro_f1_subset": 0.0,
  "prototype_usage": {
   "n_seen": 45,
   "assign_per_class_proto": [
    [
 

In [11]:
# ---- FAIL-FAST tổng kết pilot ----
ok = True
def check(name: str, cond: bool, detail: str = "") -> None:
    global ok
    print(("PASS" if cond else "FAIL"), name, detail)
    ok = ok and cond

check("B1 epoch hợp lý (< 900s)", all(h["epoch_sec"] < 900 for h in hist_b1["history"]),
      f"{[h['epoch_sec'] for h in hist_b1['history']]}")
check("ASF init ≡ ResNet18", init_diff < 1e-3, f"max|Δ|={init_diff:.2e}")
check("gate có hoạt động (var > 0)", gate_spread > 0 or cfg_asf.asf_fixed_fusion, f"{gate_spread:.2e}")
check("CMP margin âm", delta_40 < 0, f"{delta_40:.3f}")
check("latency đo được mọi model", all(v["median_ms"] > 0 for v in lats.values()))
print("PILOT", "PASS — dùng số liệu chốt compute_budget.md" if ok else "CÓ FAIL — sửa trước khi sang giai đoạn C")

PASS B1 epoch hợp lý (< 900s) [10.06, 9.71]
PASS ASF init ≡ ResNet18 max|Δ|=0.00e+00
PASS gate có hoạt động (var > 0) 2.39e-05
PASS CMP margin âm -0.240
PASS latency đo được mọi model 
PILOT PASS — dùng số liệu chốt compute_budget.md
